# EAQ — Kaggle GPU preflight

Validate the runtime before downloading LaWAM weights. This notebook checks CUDA,
SDPA precision, headless MuJoCo rendering, and access to pinned checkpoint metadata.
It does **not** load LaWAM, execute LIBERO, or report policy performance.

## Before running

1. Push this notebook, `scripts/cloud_preflight.py`, and `requirements/preflight.txt` to GitHub.
2. Create/import this notebook in Kaggle. Enable **Internet** and select a **GPU** accelerator.
3. In Kaggle **Add-ons → Secrets**, add `HF_TOKEN` and enable notebook access to it.
   The token needs read access. Your Hugging Face account must have approved access to
   [DINOv3](https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m).
4. Set `EAQ_REF` below to the full commit SHA you pushed for a reproducible run.

The repository must be publicly readable for this launcher. For a private repository,
use a separate authenticated checkout; do not paste credentials into this notebook.
No full weights or datasets are downloaded. Dependency installation needs Internet.


In [ ]:
from pathlib import Path
import datetime as dt
import json
import os
import subprocess
import sys

EAQ_REPO = "https://github.com/vmanvs/EAQ.git"
EAQ_REF = "main"  # Replace with the full pushed commit SHA for repeatable runs.
ROOT = Path("/kaggle/working/eaq-preflight")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN = ROOT / "runs" / RUN_ID
RUN.mkdir(parents=True, exist_ok=False)
CHECKOUT = ROOT / "checkouts" / RUN_ID

def run_command(args, **kwargs):
    return subprocess.run([str(arg) for arg in args], check=True, **kwargs)

print("Run directory:", RUN)


## 1. Fetch a fixed snapshot

The requested ref is resolved once and checked out detached. Its full commit is
saved in the run manifest. Rerunning this cell does not pull changes into an existing checkout.


In [ ]:
if not CHECKOUT.exists():
    CHECKOUT.parent.mkdir(parents=True, exist_ok=True)
    run_command(["git", "init", CHECKOUT])
    run_command(["git", "-C", CHECKOUT, "remote", "add", "origin", EAQ_REPO])
    run_command(["git", "-C", CHECKOUT, "fetch", "--depth", "1", "origin", EAQ_REF])
    run_command(["git", "-C", CHECKOUT, "checkout", "--detach", "FETCH_HEAD"])
commit = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip()
manifest = {"eaq_commit": commit, "requested_ref": EAQ_REF, "run_id": RUN_ID,
            "scope": "infrastructure preflight only"}
(RUN / "manifest.json").write_text(json.dumps(manifest, indent=2))
assert (CHECKOUT / "scripts/cloud_preflight.py").is_file(), "Push the preflight files before running."
print("EAQ commit:", commit)


## 2. Prepare a small environment

Reuse Kaggle's installed PyTorch/CUDA stack through a virtual environment with
system packages enabled. Install only preflight dependencies into that environment.
The complete LaWAM environment will be prepared separately after this gate passes.


In [ ]:
ENV = ROOT / "preflight-env"
PYTHON = ENV / "bin" / "python"
if not PYTHON.exists():
    run_command([sys.executable, "-m", "venv", "--system-site-packages", ENV])
run_command([PYTHON, "-m", "pip", "install", "--disable-pip-version-check",
             "-r", CHECKOUT / "requirements/preflight.txt"])


## 3. Read the access token privately

The token is passed only through the child process environment, never written to
the checkout or report. Without a secret, public metadata checks can still pass;
the gated DINOv3 check will explain the missing access.


In [ ]:
child_env = os.environ.copy()
child_env["MUJOCO_GL"] = "egl"
child_env["HF_HUB_DISABLE_TELEMETRY"] = "1"
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None
    print("HF_TOKEN secret unavailable. DINOv3 access may fail; no weights will be downloaded.")
if hf_token:
    child_env["HF_TOKEN"] = hf_token
del hf_token


## 4. Run validation

Checks run independently and write `report.json` after each completion. A failure
does not discard earlier results. GPU 0 is used for attention; a second GPU's memory
is not automatically available to the same model. Runtime timing is diagnostic only.


In [ ]:
with (RUN / "preflight.log").open("w") as log:
    process = subprocess.Popen(
        [str(PYTHON), "-u", str(CHECKOUT / "scripts/cloud_preflight.py"), "--output", str(RUN)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=child_env,
    )
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    exit_code = process.wait()
child_env.pop("HF_TOKEN", None)
print("Validation exit code:", exit_code)


## 5. Inspect and preserve the results

Review the table and rendered frame. Save a Kaggle notebook version with output
files, or download the ZIP below before stopping the session. The working directory
alone is not durable storage. Do not publish raw account-specific diagnostics
without reviewing them.


In [ ]:
import shutil
from IPython.display import display, FileLink, Image

report_path = RUN / "report.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    for name, item in report["checks"].items():
        print(f"{name:20s} {item['status']}")
        if item["status"] == "failed":
            print(" ", item["error"])
    print(json.dumps(report, indent=2))
else:
    report = {"passed": False}
    print("The process exited before writing a report; inspect preflight.log.")
if (RUN / "render.png").exists():
    display(Image(filename=str(RUN / "render.png")))
archive = shutil.make_archive(str(ROOT / f"preflight-{RUN_ID}"), "zip", RUN)
display(FileLink(archive))
assert exit_code == 0 and report["passed"], "Preflight incomplete. Preserve the report/logs and resolve failed checks."


## What a pass means

This session can execute a small GPU attention calculation, render a simple
physics scene, and access the required model metadata. It does **not** establish
that LaWAM fits, that all its operators are compatible, or that FP16 reproduces
the released BF16 policy. On T4/P100, those precision changes still need testing.

The next milestone is a separately validated LaWAM environment, one action
prediction, and then one LIBERO rollout. Keep the report's commit and environment
inventory with subsequent results.
